In [ ]:
#@title Gemini Live Transcriber { display-mode: "form" }
# One run. Prompts: model -> API key -> upload -> automatic transcription.

import os, sys, re, math, time, json, shutil, asyncio, subprocess, getpass, zipfile
from array import array
from pathlib import Path
from datetime import datetime
from google.colab import files

SAMPLE_RATE = 16_000
BYTES_PER_SAMPLE = 2
BYTES_PER_SEC = SAMPLE_RATE * BYTES_PER_SAMPLE
FRAME_MS = 100
FRAME_BYTES = BYTES_PER_SEC * FRAME_MS // 1000
OVERLAP_SEC = 1.0
TMP = Path('/content/gemini_live_tmp')
TMP.mkdir(parents=True, exist_ok=True)

MODELS = {
    '1': {
        'name': 'Gemini 3.5 Transcribe Live',
        'id': 'gemini-3.5-transcribe-live',
        'tpm': 20_000,
        'concurrency': 12,
        'max_chunk_sec': 540,
    },
    '2': {
        'name': 'Gemini 3.8 Live',
        'id': 'gemini-3.8-live',
        'tpm': 65_000,
        'concurrency': 40,
        'max_chunk_sec': 480,
    },
}


def choose_models():
    print('Choose model:')
    print('  1 = Gemini 3.5 Transcribe Live')
    print('  2 = Gemini 3.8 Live')
    print('  3 = Both (3.5 first, then 3.8)')
    while True:
        choice = input('Model [1]: ').strip() or '1'
        if choice in ('1', '2', '3'):
            return ['1', '2'] if choice == '3' else [choice]
        print('Type 1, 2, or 3.')


def ask_key():
    while True:
        key = getpass.getpass('Gemini API key: ').strip()
        if key:
            return key
        print('API key cannot be empty.')


def upload_one_file():
    print('\nChoose the recording to upload...')
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('No file was uploaded.')
    if len(uploaded) != 1:
        raise RuntimeError('Please upload exactly one recording.')
    name, data = next(iter(uploaded.items()))
    path = Path('/content') / Path(name).name
    path.write_bytes(data)
    return path


def install_sdk():
    print('\n[setup] Preparing Gemini SDK...', flush=True)
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'google-genai==2.25.0'],
        check=True,
    )
    global genai, types
    from google import genai as _genai
    from google.genai import types as _types
    genai, types = _genai, _types


def ffmpeg_to_pcm(src, dst):
    if not shutil.which('ffmpeg'):
        raise RuntimeError('ffmpeg is unavailable in this Colab runtime.')
    subprocess.run([
        'ffmpeg', '-hide_banner', '-loglevel', 'error', '-y',
        '-i', str(src), '-vn', '-ac', '1', '-ar', str(SAMPLE_RATE),
        '-f', 's16le', str(dst),
    ], check=True)
    if not dst.exists() or dst.stat().st_size == 0:
        raise RuntimeError('Audio conversion failed or produced an empty file.')
    return dst.stat().st_size / BYTES_PER_SEC


def hms(sec):
    sec = max(0, int(round(sec)))
    h, rem = divmod(sec, 3600)
    m, s = divmod(rem, 60)
    return f'{h:02d}:{m:02d}:{s:02d}'


def safe_name(s):
    return re.sub(r'[^\w.-]+', '_', s, flags=re.UNICODE).strip('_') or 'transcript'


def make_chunks(total_sec, spec):
    c = min(spec['concurrency'], max(1, math.ceil(total_sec / 2)))
    waves = max(1, math.ceil(total_sec / (c * spec['max_chunk_sec'])))
    count = min(max(1, waves * c), max(1, math.ceil(total_sec / 2)))
    chunk_len = total_sec / count
    out = []
    for i in range(count):
        nominal_start = i * chunk_len
        nominal_end = total_sec if i == count - 1 else (i + 1) * chunk_len
        out.append({
            'index': i,
            'start_sec': max(0.0, nominal_start - (OVERLAP_SEC if i else 0.0)),
            'end_sec': min(total_sec, nominal_end + (OVERLAP_SEC if i < count - 1 else 0.0)),
        })
    return out


def norm_word(w):
    return re.sub(r'[^\w\u0600-\u06FF]+', '', w, flags=re.UNICODE).casefold()


def merge_overlap(parts, max_words=60):
    merged = []
    for text in parts:
        words = text.strip().split()
        if not words:
            continue
        if not merged:
            merged = words
            continue
        left = [norm_word(x) for x in merged]
        right = [norm_word(x) for x in words]
        best = 0
        for n in range(min(max_words, len(merged), len(words)), 1, -1):
            if left[-n:] == right[:n] and any(left[-n:]):
                best = n
                break
        merged.extend(words[best:])
    return ' '.join(merged).strip()


def probably_silent(path, start_byte, end_byte, threshold=120):
    span = max(0, end_byte - start_byte)
    if span <= 0:
        return True
    window = min(BYTES_PER_SEC // 2, span)
    positions = [start_byte] if span <= window else [
        start_byte + int((span - window) * i / 7) for i in range(8)
    ]
    peak = 0.0
    with open(path, 'rb') as f:
        for pos in positions:
            pos -= pos % 2
            f.seek(pos)
            data = f.read(window)
            if len(data) < 2:
                continue
            samples = array('h')
            samples.frombytes(data[:len(data) - (len(data) % 2)])
            if sys.byteorder != 'little':
                samples.byteswap()
            if samples:
                rms = math.sqrt(sum(v * v for v in samples) / len(samples))
                peak = max(peak, rms)
    return peak < threshold


def model_config(key):
    if key == '1':
        return {
            'response_modalities': ['TEXT'],
            'input_audio_transcription': {
                'language_codes': [],
                'mode': 'VERBATIM',
            },
        }
    return {
        'response_modalities': ['AUDIO'],
        'input_audio_transcription': {},
    }


async def receive_transcript(session, stream_done):
    pieces = []
    iterator = session.receive().__aiter__()
    last_text_at = time.monotonic()
    while True:
        timeout = 30.0 if not stream_done.is_set() else (4.0 if pieces else 20.0)
        try:
            msg = await asyncio.wait_for(iterator.__anext__(), timeout=timeout)
        except (StopAsyncIteration, asyncio.TimeoutError):
            break
        sc = getattr(msg, 'server_content', None)
        tr = getattr(sc, 'input_transcription', None) if sc else None
        text = (getattr(tr, 'text', '') or '').strip() if tr else ''
        if text:
            if not pieces or text != pieces[-1]:
                pieces.append(text)
            last_text_at = time.monotonic()
        if stream_done.is_set() and pieces and time.monotonic() - last_text_at > 2.0:
            break
    return ' '.join(pieces).strip()


async def transcribe_chunk(client, pcm_path, chunk, model_key, max_attempts=3):
    spec = MODELS[model_key]
    start_byte = int(chunk['start_sec'] * BYTES_PER_SEC)
    end_byte = int(chunk['end_sec'] * BYTES_PER_SEC)
    start_byte -= start_byte % 2
    end_byte -= end_byte % 2
    total = max(0, end_byte - start_byte)
    last_error = None

    for attempt in range(1, max_attempts + 1):
        try:
            async with client.aio.live.connect(model=spec['id'], config=model_config(model_key)) as session:
                done = asyncio.Event()
                recv = asyncio.create_task(receive_transcript(session, done))
                remaining = total
                with open(pcm_path, 'rb', buffering=1024 * 1024) as f:
                    f.seek(start_byte)
                    next_send = time.monotonic()
                    while remaining > 0:
                        data = f.read(min(FRAME_BYTES, remaining))
                        if not data:
                            break
                        await session.send_realtime_input(
                            audio=types.Blob(data=data, mime_type='audio/pcm;rate=16000')
                        )
                        remaining -= len(data)
                        next_send += FRAME_MS / 1000.0
                        delay = next_send - time.monotonic()
                        if delay > 0:
                            await asyncio.sleep(delay)
                await session.send_realtime_input(audio_stream_end=True)
                done.set()
                try:
                    text = await asyncio.wait_for(recv, timeout=30.0)
                finally:
                    if not recv.done():
                        recv.cancel()
                if not text and not await asyncio.to_thread(
                    probably_silent, pcm_path, start_byte, end_byte
                ):
                    raise RuntimeError('empty transcript on non-silent audio')
                return text
        except Exception as e:
            last_error = e
            if attempt < max_attempts:
                await asyncio.sleep(2 * attempt)
    raise RuntimeError(f'chunk {chunk["index"] + 1} failed: {last_error}')


async def transcribe_model(api_key, pcm_path, duration, model_key):
    spec = MODELS[model_key]
    chunks = make_chunks(duration, spec)
    concurrency = min(spec['concurrency'], len(chunks))
    results = [None] * len(chunks)
    client = genai.Client(api_key=api_key)
    started = time.monotonic()
    done_count = 0
    stop_heartbeat = asyncio.Event()

    expected = max(ch['end_sec'] - ch['start_sec'] for ch in chunks) * math.ceil(len(chunks) / concurrency)
    print(
        f'\n[{spec["name"]}] {len(chunks)} chunks | up to {concurrency} parallel | '
        f'audio {hms(duration)} | rough minimum ~{expected/60:.1f} min',
        flush=True,
    )

    async def heartbeat():
        while not stop_heartbeat.is_set():
            elapsed = time.monotonic() - started
            print(
                f'\r[{spec["name"]}] working... {done_count}/{len(chunks)} chunks | elapsed {hms(elapsed)}',
                end='', flush=True,
            )
            try:
                await asyncio.wait_for(stop_heartbeat.wait(), timeout=10)
            except asyncio.TimeoutError:
                pass

    hb = asyncio.create_task(heartbeat())

    async def worker(ch):
        nonlocal done_count
        text = await transcribe_chunk(client, pcm_path, ch, model_key)
        results[ch['index']] = text
        done_count += 1

    async def run_wave(wave):
        tasks = [asyncio.create_task(worker(ch)) for ch in wave]
        settled = await asyncio.gather(*tasks, return_exceptions=True)
        return [x for x in settled if isinstance(x, Exception)]

    try:
        failures = []
        for offset in range(0, len(chunks), concurrency):
            failures.extend(await run_wave(chunks[offset:offset + concurrency]))

        missing = [ch for ch in chunks if results[ch['index']] is None]
        recovery = max(1, concurrency // 2)
        round_no = 0
        while missing and round_no < 5:
            round_no += 1
            print(f'\n[{spec["name"]}] retrying {len(missing)} failed chunk(s) with concurrency {recovery}...', flush=True)
            for offset in range(0, len(missing), recovery):
                await run_wave(missing[offset:offset + recovery])
            missing = [ch for ch in chunks if results[ch['index']] is None]
            recovery = max(1, recovery // 2)

        if missing:
            raise RuntimeError(
                f'{len(missing)} chunk(s) still failed after retries; no incomplete transcript was saved.'
            )

        transcript = merge_overlap(results)
        elapsed = time.monotonic() - started
        print(f'\r[{spec["name"]}] done: {len(chunks)}/{len(chunks)} chunks | {elapsed/60:.2f} min' + ' ' * 20, flush=True)
        return transcript, elapsed
    finally:
        stop_heartbeat.set()
        try:
            await hb
        except Exception:
            pass
        try:
            client.close()
        except Exception:
            pass


async def main():
    chosen = choose_models()
    api_key = ask_key()
    src = upload_one_file()
    install_sdk()

    pcm = TMP / f'{int(time.time())}_{safe_name(src.stem)}.pcm'
    print('\n[1/3] Preparing audio...', flush=True)
    prep_start = time.monotonic()
    duration = await asyncio.to_thread(ffmpeg_to_pcm, src, pcm)
    print(f'[1/3] Ready: {src.name} | duration {hms(duration)} | prep {time.monotonic()-prep_start:.1f}s', flush=True)

    stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    base = safe_name(src.stem)
    outputs = []
    timings = []

    try:
        print('[2/3] Transcribing...', flush=True)
        for key in chosen:
            text, elapsed = await transcribe_model(api_key, pcm, duration, key)
            spec = MODELS[key]
            out = Path('/content') / f'{base}_{spec["id"]}_{stamp}.txt'
            out.write_text(text, encoding='utf-8')
            outputs.append(out)
            timings.append((spec['name'], elapsed))

        print('\n[3/3] Finished.', flush=True)
        for name, elapsed in timings:
            print(f'  {name}: {elapsed/60:.2f} min')

        if len(outputs) == 1:
            print(f'\nDownloading: {outputs[0].name}', flush=True)
            files.download(str(outputs[0]))
        else:
            z = Path('/content') / f'{base}_Gemini_Live_transcripts_{stamp}.zip'
            with zipfile.ZipFile(z, 'w', zipfile.ZIP_DEFLATED) as archive:
                for p in outputs:
                    archive.write(p, arcname=p.name)
            print(f'\nDownloading: {z.name}', flush=True)
            files.download(str(z))
    finally:
        try:
            pcm.unlink(missing_ok=True)
        except Exception:
            pass


await main()


In [ ]:
#@title Gemini 3.5 Transcribe Live Benchmark { display-mode: "form" }
# Official-config correctness check -> clean adaptive concurrency -> sustained validation.

import sys, math, time, asyncio, subprocess, getpass, shutil
from array import array
from pathlib import Path
from collections import Counter
from google.colab import files

MODEL_NAME = "Gemini 3.5 Transcribe Live"
MODEL_ID = "gemini-3.5-transcribe-live"
SAMPLE_RATE = 16_000
BYTES_PER_SAMPLE = 2
BYTES_PER_SEC = SAMPLE_RATE * BYTES_PER_SAMPLE
FRAME_MS = 100
FRAME_BYTES = BYTES_PER_SEC * FRAME_MS // 1000

PROBE_SEC = 20
VALIDATE_SEC = 60
PROFILES = (2, 4, 6, 8, 10, 12)
COOLDOWN_SEC = 45
TMP = Path("/content/gemini_35_live_benchmark")
TMP.mkdir(parents=True, exist_ok=True)


def ask_key():
    while True:
        key = getpass.getpass("Gemini API key: ").strip()
        if key:
            return key
        print("API key cannot be empty.")


def upload_one_file():
    print("\nChoose the recording to upload...")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file was uploaded.")
    if len(uploaded) != 1:
        raise RuntimeError("Please upload exactly one recording.")
    name, data = next(iter(uploaded.items()))
    path = Path("/content") / Path(name).name
    path.write_bytes(data)
    return path


def install_sdk():
    print("\n[setup] Preparing Gemini SDK...", flush=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "google-genai==2.25.0"],
        check=True,
    )
    global genai, types
    from google import genai as _genai
    from google.genai import types as _types
    genai, types = _genai, _types


def ffmpeg_to_pcm(src, dst):
    if not shutil.which("ffmpeg"):
        raise RuntimeError("ffmpeg is unavailable in this Colab runtime.")
    subprocess.run(
        [
            "ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
            "-i", str(src), "-vn", "-ac", "1", "-ar", str(SAMPLE_RATE),
            "-f", "s16le", str(dst),
        ],
        check=True,
    )
    if not dst.exists() or dst.stat().st_size == 0:
        raise RuntimeError("Audio conversion failed or produced an empty file.")
    return dst.stat().st_size / BYTES_PER_SEC


def live_config():
    # Mirrors Google's current documented Python configuration for Live Transcription.
    return types.LiveConnectConfig(
        response_modalities=["TEXT"],
        input_audio_transcription=types.AudioTranscriptionConfig(
            language_codes=[],
            mode="VERBATIM",
        ),
    )


def rms_at(path, start_sec, span_sec):
    start = int(start_sec * BYTES_PER_SEC) // 2 * 2
    size = int(span_sec * BYTES_PER_SEC) // 2 * 2
    with open(path, "rb") as f:
        f.seek(start)
        data = f.read(size)
    samples = array("h")
    samples.frombytes(data[: len(data) - (len(data) % 2)])
    if sys.byteorder != "little":
        samples.byteswap()
    if not samples:
        return 0.0
    return math.sqrt(sum(v * v for v in samples) / len(samples))


def choose_probe(pcm_path, duration, seconds):
    span = min(float(seconds), duration)
    if duration <= span:
        return 0.0, span
    max_start = duration - span
    starts = [max_start * p for p in (0.10, 0.30, 0.50, 0.70, 0.90)]
    _, start = max((rms_at(pcm_path, s, span), s) for s in starts)
    return start, span


def append_unique(parts, text):
    text = (text or "").strip()
    if text and (not parts or text != parts[-1]):
        parts.append(text)


async def receive_transcript(session, stream_done):
    finals = []
    interims = []
    events = Counter()
    iterator = session.receive().__aiter__()
    last_activity = time.monotonic()

    while True:
        timeout = 25.0 if not stream_done.is_set() else 8.0
        try:
            response = await asyncio.wait_for(iterator.__anext__(), timeout=timeout)
        except StopAsyncIteration:
            events["receive_end"] += 1
            break
        except asyncio.TimeoutError:
            events["receive_timeout"] += 1
            break

        last_activity = time.monotonic()

        if getattr(response, "setup_complete", None):
            events["setup_complete"] += 1

        sc = getattr(response, "server_content", None)
        if not sc:
            events["other_message"] += 1
            continue

        interim = getattr(sc, "interim_input_transcription", None)
        final = getattr(sc, "input_transcription", None)

        if interim:
            events["interim_input_transcription"] += 1
            append_unique(interims, getattr(interim, "text", ""))

        if final:
            events["input_transcription"] += 1
            append_unique(finals, getattr(final, "text", ""))

        if getattr(sc, "turn_complete", False):
            events["turn_complete"] += 1

        if stream_done.is_set() and finals and time.monotonic() - last_activity > 2.0:
            break

    return " ".join(finals).strip(), " ".join(interims).strip(), events


async def stream_probe(session, pcm_path, start_sec, seconds):
    start_byte = int(start_sec * BYTES_PER_SEC) // 2 * 2
    remaining = int(seconds * BYTES_PER_SEC) // 2 * 2

    with open(pcm_path, "rb", buffering=1024 * 1024) as f:
        f.seek(start_byte)
        next_send = time.monotonic()

        while remaining > 0:
            data = f.read(min(FRAME_BYTES, remaining))
            if not data:
                break

            await session.send_realtime_input(
                audio=types.Blob(
                    data=data,
                    mime_type="audio/pcm;rate=16000",
                )
            )
            remaining -= len(data)

            next_send += FRAME_MS / 1000.0
            delay = next_send - time.monotonic()
            if delay > 0:
                await asyncio.sleep(delay)

    await session.send_realtime_input(audio_stream_end=True)


async def one_probe(client, pcm_path, start_sec, seconds):
    started = time.monotonic()
    recv = None

    try:
        async with client.aio.live.connect(
            model=MODEL_ID,
            config=live_config(),
        ) as session:
            stream_done = asyncio.Event()
            recv = asyncio.create_task(receive_transcript(session, stream_done))

            await stream_probe(session, pcm_path, start_sec, seconds)
            stream_done.set()

            final, interim, events = await asyncio.wait_for(recv, timeout=20.0)

            if not final:
                kind = "interim-only transcript" if interim else "empty transcript"
                return {
                    "ok": False,
                    "elapsed": time.monotonic() - started,
                    "error": kind,
                    "final": final,
                    "interim": interim,
                    "events": events,
                }

            return {
                "ok": True,
                "elapsed": time.monotonic() - started,
                "error": "",
                "final": final,
                "interim": interim,
                "events": events,
            }

    except Exception as e:
        return {
            "ok": False,
            "elapsed": time.monotonic() - started,
            "error": f"{type(e).__name__}: {e}",
            "final": "",
            "interim": "",
            "events": Counter(),
        }

    finally:
        if recv is not None and not recv.done():
            recv.cancel()


async def run_parallel(client, pcm_path, start_sec, seconds, concurrency):
    started = time.monotonic()
    results = await asyncio.gather(
        *(
            one_probe(client, pcm_path, start_sec, seconds)
            for _ in range(concurrency)
        )
    )
    elapsed = time.monotonic() - started
    successes = sum(r["ok"] for r in results)
    errors = Counter(r["error"] for r in results if not r["ok"])
    speed = (concurrency * seconds / elapsed) if elapsed else float("inf")
    return results, successes, elapsed, speed, errors


def quota_like(errors):
    text = " ".join(errors).casefold()
    return any(token in text for token in ("quota", "resource_exhausted", "429", "1011"))


def show_single(result):
    print(f'  status: {"PASS" if result["ok"] else "FAIL"} | {result["elapsed"]:.1f}s')
    if result["events"]:
        print("  events:", ", ".join(f"{k}={v}" for k, v in result["events"].items()))
    if result["final"]:
        print("  final sample:", result["final"][:220].replace("\n", " "))
    elif result["interim"]:
        print("  interim sample:", result["interim"][:220].replace("\n", " "))
    if result["error"]:
        print("  error:", result["error"][:350])


def print_row(concurrency, successes, elapsed, speed, errors):
    top_error = errors.most_common(1)[0][0] if errors else ""
    if len(top_error) > 123:
        top_error = top_error[:120] + "..."
    print(
        f"  {concurrency:>11} | {successes:>2}/{concurrency:<2}   | "
        f"{concurrency-successes:>6} | {elapsed:>6.1f}s | "
        f"{speed:>7.2f}x realtime | {top_error}"
    )


async def benchmark(api_key, pcm_path, duration):
    client = genai.Client(api_key=api_key)

    try:
        short_start, short_sec = choose_probe(pcm_path, duration, PROBE_SEC)

        print(
            f"\n[{MODEL_NAME}] source={duration/60:.2f} min | "
            f"short probe={short_sec:.0f}s @ {short_start:.1f}s",
            flush=True,
        )

        print("\nPhase A - official single-session correctness check")
        single = await one_probe(client, pcm_path, short_start, short_sec)
        show_single(single)

        if not single["ok"]:
            print("\nSTOP: 1/1 failed, so concurrency was not tested.")
            return

        passed = [1]

        print("\nPhase B - clean adaptive concurrency")
        print(f"  ({COOLDOWN_SEC}s cooldown keeps profile starts more than a minute apart.)")
        print("  concurrency | success | failed | elapsed | effective speed | top error")
        print("  ------------+---------+--------+---------+-----------------+----------")
        print_row(1, 1, single["elapsed"], short_sec / single["elapsed"], Counter())

        for concurrency in PROFILES:
            print(f"\n  cooling down {COOLDOWN_SEC}s before concurrency {concurrency}...", flush=True)
            await asyncio.sleep(COOLDOWN_SEC)

            _, successes, elapsed, speed, errors = await run_parallel(
                client, pcm_path, short_start, short_sec, concurrency
            )
            print_row(concurrency, successes, elapsed, speed, errors)

            if successes == concurrency:
                passed.append(concurrency)
                continue

            if quota_like(errors):
                print(f"\n  Stop reason: quota/capacity error at concurrency {concurrency}.")
            else:
                print(f"\n  Stop reason: concurrency {concurrency} was not 100% successful.")
            break

        candidate = max(passed)

        print(
            f"\nPhase C - sustained validation of concurrency {candidate} "
            f"for {VALIDATE_SEC}s/session"
        )

        long_start, long_sec = choose_probe(pcm_path, duration, VALIDATE_SEC)
        await asyncio.sleep(COOLDOWN_SEC)

        _, successes, elapsed, speed, errors = await run_parallel(
            client, pcm_path, long_start, long_sec, candidate
        )
        print("  concurrency | success | failed | elapsed | effective speed | top error")
        print("  ------------+---------+--------+---------+-----------------+----------")
        print_row(candidate, successes, elapsed, speed, errors)

        if successes != candidate:
            print(
                "\nRESULT: short-burst concurrency passed, but the sustained check failed. "
                "Do not change Production yet."
            )
            return

        ideal_minutes = duration / candidate / 60.0
        print(f"\nRESULT: validated concurrency = {candidate}")
        print(
            f"Ideal streaming floor for this {duration/60:.2f}-min recording: "
            f"~{ideal_minutes:.2f} min + finalization/retry overhead."
        )

    finally:
        try:
            client.close()
        except Exception:
            pass


async def main():
    print("Gemini 3.5 Transcribe Live benchmark")
    print("Uses Google's documented TEXT + input_audio_transcription configuration.")
    api_key = ask_key()
    src = upload_one_file()
    install_sdk()

    pcm = TMP / f"{int(time.time())}_benchmark.pcm"

    try:
        print("\nPreparing audio...", flush=True)
        duration = await asyncio.to_thread(ffmpeg_to_pcm, src, pcm)
        print(f"Ready: {src.name} | duration {duration/60:.2f} min", flush=True)
        await benchmark(api_key, pcm, duration)
    finally:
        try:
            pcm.unlink(missing_ok=True)
        except Exception:
            pass


await main()
